In [ ]:
import pandas as pd
import os

In [112]:
# file_path = "D:/skn31_2nd_pr/data/test.csv"
# df = pd.read_csv(file_path)
df = pd.read_csv("D:/skn31_2nd_pr/data/test_원본.csv")
df.head()

,msno,is_churn,city,registered_via,registration_init_time,reg_year,reg_month,reg_day,reg_weekday,reg_time_num,...,trans_count,total_secs,num_25,num_50,num_75,num_985,num_100,num_unq,log_count,last_listen_date
0,k14Hn0yxprq4DqaVzCyLaDE90wNilMRck43aaxc3lps=,0,1,7,2016-04-18,2016,4,18,0,2016.299315,...,12,689923.30,208,58,48,43,2507,2177,31,20170331
1,I50pzedPKCjPIBf9xMtLvbZm9yv37YlIfgoubYgdMPU=,0,18,7,2012-08-31,2012,8,31,4,2012.668265,...,28,211559.97,5,5,5,4,836,796,16,20170326
2,x+mwAUyTwpl5gLaUAZXEv5KMmrtRpMUNqQbKAk20x1A=,0,1,7,2016-02-26,2016,2,26,4,2016.154566,...,14,404254.30,7,1,0,1,1841,635,31,20170331
3,qT6aYgjNuEQaYq7zzrbv3pUlcbmJk5CaHEnN5ZeEeHo=,0,1,7,2013-08-17,2013,8,17,5,2013.629909,...,27,108673.11,79,24,20,14,348,384,20,20170331
4,lE5bxjL17l7uIEjBdHr7Pqal/mQ42+fvbZBqXfwOLNY=,0,8,4,2015-11-18,2015,11,18,2,2015.882648,...,10,246419.61,323,128,41,51,921,1027,30,20170331


In [108]:
data[data['msno'] == 0]

,msno,actual_churn,predicted_churn,churn_probability,risk_decile,risk_group


# msno 변경

In [ ]:
import pandas as pd

file_path = "D:/skn31_2nd_pr/data/test_원본.csv"
df = pd.read_csv(file_path)

# 1. 고유한 유저(msno) 목록만 추출해서 정렬
unique_users = df['msno'].unique()

# 2. '지저분한 문자열' : '깔끔한 ID' 형태의 딕셔너리(매핑 가이드) 생성
# 예: {'f/NmvEzHfh...': 'user_00001', 'zLo9f73nGG...': 'user_00002'}
user_mapping = {
    raw_msno: f"user_{i+1:06d}"  # 05d는 5자리 숫자로 맞춰줍니다 (user_00001, user_00002 ...)
    for i, raw_msno in enumerate(unique_users)
}

# 3. 데이터프레임의 기존 msno 값을 딕셔너리 기준으로 한 번에 치환 (.map 함수 사용)
df['msno'] = df['msno'].map(user_mapping)

# 3. 컬럼명 강제 변환 ('msno' -> '고객_ID')
# 혹시 대소문자가 섞여있을 상황을 대비해 매핑을 걸어줍니다.
df.rename(columns={"msno": "고객_ID", "MSNO": "고객_ID"}, inplace=True)

# 4. 원본 경로에 그대로 인덱스 없이 덮어씌우기 (인코딩 utf-8-sig로 한글 깨짐 방지)
df.to_csv(file_path, index=False, encoding="utf-8-sig")

print(f"🎯 컬럼명 변환 및 원본 덮어쓰기 완료! 경로: {file_path}")

🎯 컬럼명 변환 및 원본 덮어쓰기 완료! 경로: D:/skn31_2nd_pr/data/test_원본.csv


# 2, 컬럼명 일괄 변경 

In [137]:
import pandas as pd

def translate_features(df_feat):
    feature_map = {
        "trans_day": "결제일(일)", "expire_day": "만료일(일)",
        "trans_weekday": "결제요일", "expire_weekday": "만료요일",
        "trans_month": "결제월", "expire_month": "만료월",
        "trans_year": "결제연도", "expire_year": "만료연도",
        "reg_time_num": "가입기간(일)", "reg_day": "가입일(일)",
        "reg_month": "가입월", "reg_year": "가입연도", "reg_weekday": "가입요일",
        "city": "거주도시", "registered_via": "가입채널",
        "num_25": "스킵 청취 곡수(25% 미만)", "num_50": "단기 청취 곡수(50%)",
        "num_75": "중기 청취 곡수(75%)", "num_985": "장기 청취 곡수(98.5%)",
        "num_100": "완독 청취 곡수(100%)", "num_unq": "고유 재생 곡수",
        "total_secs": "총청취시간(초)", "log_count": "월간접속일수",
        "last_listen_date": "마지막청취일", "payment_method_id": "결제수단ID",
        "trans_count": "누적결제_횟수", "is_auto_renew": "자동갱신_동의여부",
        "plan_list_price": "구독 정가", "actual_amount_paid": "실제_결제금액",
        "is_cancel": "구독_직접_취소여부", "tx_payment_plan_days": "플랜이용기간",
        "gender_male": "성별: 남성", "gender_female": "성별: 여성", "gender_unknown": "성별: 미입력",
        "pm_id_38": "결제수단ID_31번", "pm_id_41": "결제수단ID_34번",
        "pp_days_30": "30일_정기구독", "is_churn":"이탈여부","registration_init_time":"최초가입날짜","trans_count":"누적결제수",
        "membership_expire_date":"멤버십종료일","transaction_date":"가장최근결제날짜","payment_plan_days":"구독플랜일수"
    }
    
    df_clean = df.copy()
        
        # 🔍 공백으로 인한 매핑 실패를 방지하기 위해 현재 컬럼명의 좌우 공백 제거(strip)
    df_clean.columns = [c.strip() for c in df_clean.columns]
        
        # 🎯 전역 매핑 사전을 적용하여 컬럼명 강제 변경 (매핑 안 된 컬럼은 원본 이름 유지)
    df_clean.rename(columns=feature_map, inplace=True)
        
    return df_clean

In [138]:
df = translate_features(df)

In [139]:
df.head()

,고객_ID,이탈여부,거주도시,가입채널,최초가입날짜,가입연도,가입월,가입일(일),가입요일,가입기간(일),...,누적결제수,총청취시간(초),스킵 청취 곡수(25% 미만),단기 청취 곡수(50%),중기 청취 곡수(75%),장기 청취 곡수(98.5%),완독 청취 곡수(100%),고유 재생 곡수,월간접속일수,마지막청취일
0,user_000001,0,1,7,2016-04-18,2016,4,18,0,2016.299315,...,12,689923.30,208,58,48,43,2507,2177,31,20170331
1,user_000002,0,18,7,2012-08-31,2012,8,31,4,2012.668265,...,28,211559.97,5,5,5,4,836,796,16,20170326
2,user_000003,0,1,7,2016-02-26,2016,2,26,4,2016.154566,...,14,404254.30,7,1,0,1,1841,635,31,20170331
3,user_000004,0,1,7,2013-08-17,2013,8,17,5,2013.629909,...,27,108673.11,79,24,20,14,348,384,20,20170331
4,user_000005,0,8,4,2015-11-18,2015,11,18,2,2015.882648,...,10,246419.61,323,128,41,51,921,1027,30,20170331


In [140]:
df.columns

Index(['고객_ID', '이탈여부', '거주도시', '가입채널', '최초가입날짜', '가입연도', '가입월', '가입일(일)',
       '가입요일', '가입기간(일)', '성별: 여성', '성별: 남성', '성별: 미입력', '결제수단ID', '구독플랜일수',
       '구독 정가', '실제_결제금액', '자동갱신_동의여부', '가장최근결제날짜', '멤버십종료일', '구독_직접_취소여부',
       '결제연도', '결제월', '결제일(일)', '결제요일', '만료연도', '만료월', '만료일(일)', '만료요일',
       '결제수단ID_34번', '결제수단ID_31번', '30일_정기구독', '누적결제수', '총청취시간(초)',
       '스킵 청취 곡수(25% 미만)', '단기 청취 곡수(50%)', '중기 청취 곡수(75%)', '장기 청취 곡수(98.5%)',
       '완독 청취 곡수(100%)', '고유 재생 곡수', '월간접속일수', '마지막청취일'],
      dtype='str')

In [141]:
# 4. 원본 경로에 그대로 인덱스 없이 덮어씌우기 (인코딩 utf-8-sig로 한글 깨짐 방지)
df.to_csv(file_path, index=False, encoding="utf-8-sig")

print(f"🎯 컬럼명 변환 및 원본 덮어쓰기 완료! 경로: {file_path}")

🎯 컬럼명 변환 및 원본 덮어쓰기 완료! 경로: D:/skn31_2nd_pr/data/test_원본.csv


In [142]:
df.head()

,고객_ID,이탈여부,거주도시,가입채널,최초가입날짜,가입연도,가입월,가입일(일),가입요일,가입기간(일),...,누적결제수,총청취시간(초),스킵 청취 곡수(25% 미만),단기 청취 곡수(50%),중기 청취 곡수(75%),장기 청취 곡수(98.5%),완독 청취 곡수(100%),고유 재생 곡수,월간접속일수,마지막청취일
0,user_000001,0,1,7,2016-04-18,2016,4,18,0,2016.299315,...,12,689923.30,208,58,48,43,2507,2177,31,20170331
1,user_000002,0,18,7,2012-08-31,2012,8,31,4,2012.668265,...,28,211559.97,5,5,5,4,836,796,16,20170326
2,user_000003,0,1,7,2016-02-26,2016,2,26,4,2016.154566,...,14,404254.30,7,1,0,1,1841,635,31,20170331
3,user_000004,0,1,7,2013-08-17,2013,8,17,5,2013.629909,...,27,108673.11,79,24,20,14,348,384,20,20170331
4,user_000005,0,8,4,2015-11-18,2015,11,18,2,2015.882648,...,10,246419.61,323,128,41,51,921,1027,30,20170331


In [143]:
df['총청취시간(분)'] = (df['총청취시간(초)'] // 60).fillna(0).astype(int)

In [145]:
df.to_csv(file_path, index=False, encoding="utf-8-sig")

print(f"🎯 컬럼명 변환 및 원본 덮어쓰기 완료! 경로: {file_path}")

🎯 컬럼명 변환 및 원본 덮어쓰기 완료! 경로: D:/skn31_2nd_pr/data/test_원본.csv


# 요일 변환

In [146]:
df.head()

,고객_ID,이탈여부,거주도시,가입채널,최초가입날짜,가입연도,가입월,가입일(일),가입요일,가입기간(일),...,총청취시간(초),스킵 청취 곡수(25% 미만),단기 청취 곡수(50%),중기 청취 곡수(75%),장기 청취 곡수(98.5%),완독 청취 곡수(100%),고유 재생 곡수,월간접속일수,마지막청취일,총청취시간(분)
0,user_000001,0,1,7,2016-04-18,2016,4,18,0,2016.299315,...,689923.30,208,58,48,43,2507,2177,31,20170331,11498
1,user_000002,0,18,7,2012-08-31,2012,8,31,4,2012.668265,...,211559.97,5,5,5,4,836,796,16,20170326,3525
2,user_000003,0,1,7,2016-02-26,2016,2,26,4,2016.154566,...,404254.30,7,1,0,1,1841,635,31,20170331,6737
3,user_000004,0,1,7,2013-08-17,2013,8,17,5,2013.629909,...,108673.11,79,24,20,14,348,384,20,20170331,1811
4,user_000005,0,8,4,2015-11-18,2015,11,18,2,2015.882648,...,246419.61,323,128,41,51,921,1027,30,20170331,4106


In [147]:
df.columns

Index(['고객_ID', '이탈여부', '거주도시', '가입채널', '최초가입날짜', '가입연도', '가입월', '가입일(일)',
       '가입요일', '가입기간(일)', '성별: 여성', '성별: 남성', '성별: 미입력', '결제수단ID', '구독플랜일수',
       '구독 정가', '실제_결제금액', '자동갱신_동의여부', '가장최근결제날짜', '멤버십종료일', '구독_직접_취소여부',
       '결제연도', '결제월', '결제일(일)', '결제요일', '만료연도', '만료월', '만료일(일)', '만료요일',
       '결제수단ID_34번', '결제수단ID_31번', '30일_정기구독', '누적결제수', '총청취시간(초)',
       '스킵 청취 곡수(25% 미만)', '단기 청취 곡수(50%)', '중기 청취 곡수(75%)', '장기 청취 곡수(98.5%)',
       '완독 청취 곡수(100%)', '고유 재생 곡수', '월간접속일수', '마지막청취일', '총청취시간(분)'],
      dtype='str')

In [148]:
def convert_num_to_weekday(df, target_column):
    """
    0~6 요일 숫자를 '월'~'일' 한글 문자열로 변환하는 함수
    """
    weekday_map = {
        0: "월요일",
        1: "화요일",
        2: "수요일",
        3: "목요일",
        4: "금요일",
        5: "토요일",
        6: "일요일"
    }
    
    # 데이터에 혹시 결측치(NaN)나 소수점(.0)형태가 섞여 있을 상황을 대비해 안전하게 매핑
    df[target_column] = df[target_column].fillna(0).astype(int).map(weekday_map).fillna(df[target_column])
    return df

df = convert_num_to_weekday(df, "가입요일")
df = convert_num_to_weekday(df, "만료요일")
df = convert_num_to_weekday(df, "결제요일")

In [149]:
df.head()

,고객_ID,이탈여부,거주도시,가입채널,최초가입날짜,가입연도,가입월,가입일(일),가입요일,가입기간(일),...,총청취시간(초),스킵 청취 곡수(25% 미만),단기 청취 곡수(50%),중기 청취 곡수(75%),장기 청취 곡수(98.5%),완독 청취 곡수(100%),고유 재생 곡수,월간접속일수,마지막청취일,총청취시간(분)
0,user_000001,0,1,7,2016-04-18,2016,4,18,월요일,2016.299315,...,689923.30,208,58,48,43,2507,2177,31,20170331,11498
1,user_000002,0,18,7,2012-08-31,2012,8,31,금요일,2012.668265,...,211559.97,5,5,5,4,836,796,16,20170326,3525
2,user_000003,0,1,7,2016-02-26,2016,2,26,금요일,2016.154566,...,404254.30,7,1,0,1,1841,635,31,20170331,6737
3,user_000004,0,1,7,2013-08-17,2013,8,17,토요일,2013.629909,...,108673.11,79,24,20,14,348,384,20,20170331,1811
4,user_000005,0,8,4,2015-11-18,2015,11,18,수요일,2015.882648,...,246419.61,323,128,41,51,921,1027,30,20170331,4106


In [150]:
df.to_csv(file_path, index=False, encoding="utf-8-sig")

print(f"🎯 컬럼명 변환 및 원본 덮어쓰기 완료! 경로: {file_path}")

🎯 컬럼명 변환 및 원본 덮어쓰기 완료! 경로: D:/skn31_2nd_pr/data/test_원본.csv


# 요금제 그룹화

In [151]:
import numpy as np
price = df['구독 정가'].astype(float)

conditions = [
    # 1. 메인 정기 구독군 (KKBOX 시그니처 149 라인업)
    (price == 149.0) | (price == 129.0) | (price == 150.0),
    
    # 2. 실속/제휴 할인군 (99~100 라인업)
    (price == 99.0) | (price == 100.0) | (price == 300.0),
    
    # 3. 고가치 장기 락인 VIP군 (400 NT$ 이상의 거대 장기 이용권)
    (price >= 400.0)
]

# 화면에 출력될 예쁜 한글 카테고리 명칭들
choices = [
    '정기구독군',
    '실속할인군',
    '고가치장기군'
]

# 위 세 개 조건에 걸리지 않는 나머지 (0원 및 소수 금액 요금) 처리
df['요금제_그룹'] = np.select(conditions, choices, default='체험·기타군')

# 3. 제대로 묶였는지 마케터 마인드로 그룹별 볼륨 전수 확인 검증
print("="*60)
print("📊 [그룹핑 검증] 요금제 카테고리별 유저 수 및 이탈률 교차 분석")
print("="*60)



📊 [그룹핑 검증] 요금제 카테고리별 유저 수 및 이탈률 교차 분석


In [153]:
df.head()

,고객_ID,이탈여부,거주도시,가입채널,최초가입날짜,가입연도,가입월,가입일(일),가입요일,가입기간(일),...,스킵 청취 곡수(25% 미만),단기 청취 곡수(50%),중기 청취 곡수(75%),장기 청취 곡수(98.5%),완독 청취 곡수(100%),고유 재생 곡수,월간접속일수,마지막청취일,총청취시간(분),요금제_그룹
0,user_000001,0,1,7,2016-04-18,2016,4,18,월요일,2016.299315,...,208,58,48,43,2507,2177,31,20170331,11498,정기구독군
1,user_000002,0,18,7,2012-08-31,2012,8,31,금요일,2012.668265,...,5,5,5,4,836,796,16,20170326,3525,실속할인군
2,user_000003,0,1,7,2016-02-26,2016,2,26,금요일,2016.154566,...,7,1,0,1,1841,635,31,20170331,6737,실속할인군
3,user_000004,0,1,7,2013-08-17,2013,8,17,토요일,2013.629909,...,79,24,20,14,348,384,20,20170331,1811,정기구독군
4,user_000005,0,8,4,2015-11-18,2015,11,18,수요일,2015.882648,...,323,128,41,51,921,1027,30,20170331,4106,정기구독군


In [154]:
df.to_csv(file_path, index=False, encoding="utf-8-sig")

print(f"🎯 컬럼명 변환 및 원본 덮어쓰기 완료! 경로: {file_path}")

🎯 컬럼명 변환 및 원본 덮어쓰기 완료! 경로: D:/skn31_2nd_pr/data/test_원본.csv


# 예측 데이터 받으면

In [155]:
import pandas as pd

file_path = "D:/skn31_2nd_pr/data/모델_예측.csv"
df = pd.read_csv(file_path)

# 1. 고유한 유저(msno) 목록만 추출해서 정렬
unique_users = df['msno'].unique()

# 2. '지저분한 문자열' : '깔끔한 ID' 형태의 딕셔너리(매핑 가이드) 생성
# 예: {'f/NmvEzHfh...': 'user_00001', 'zLo9f73nGG...': 'user_00002'}
user_mapping = {
    raw_msno: f"user_{i+1:06d}"  # 05d는 5자리 숫자로 맞춰줍니다 (user_00001, user_00002 ...)
    for i, raw_msno in enumerate(unique_users)
}

# 3. 데이터프레임의 기존 msno 값을 딕셔너리 기준으로 한 번에 치환 (.map 함수 사용)
df['msno'] = df['msno'].map(user_mapping)

# 3. 컬럼명 강제 변환 ('msno' -> '고객_ID')
# 혹시 대소문자가 섞여있을 상황을 대비해 매핑을 걸어줍니다.
df.rename(columns={"msno": "고객_ID", "MSNO": "고객_ID"}, inplace=True)

# 4. 원본 경로에 그대로 인덱스 없이 덮어씌우기 (인코딩 utf-8-sig로 한글 깨짐 방지)
df.to_csv(file_path, index=False, encoding="utf-8-sig")

print(f"🎯 컬럼명 변환 및 원본 덮어쓰기 완료! 경로: {file_path}")

🎯 컬럼명 변환 및 원본 덮어쓰기 완료! 경로: D:/skn31_2nd_pr/data/모델_예측.csv


In [158]:
import pandas as pd
import numpy as np

# 1. 경로 지정 (본인 절대경로 환경에 맞게 수정하세요)
test_path = "D:/skn31_2nd_pr/data/test_원본.csv"
pred_path = "D:/skn31_2nd_pr/data/모델_예측.csv"

print("🚀 데이터셋 병합 및 최적화(다운캐스팅) 전처리를 시작합니다...")

# 2. 데이터 로드
df_test = pd.read_csv(test_path)
df_pred = pd.read_csv(pred_path)

# 3. 예측 데이터 정제 및 컬럼명 싱크 맞추기
# msno -> 고객_ID 변환
if 'msno' in df_pred.columns:
    df_pred = df_pred.rename(columns={'msno': '고객_ID'})
elif 'MSNO' in df_pred.columns:
    df_pred = df_pred.rename(columns={'MSNO': '고객_ID'})

# actual_churn은 정답(is_churn)과 중복되므로 과감히 제외
# 필요한 컬럼만 슬라이싱 추출
pred_cols = ['고객_ID', 'predicted_churn', 'churn_probability', 'risk_decile', 'risk_group']
df_pred_clean = df_pred[[c for c in pred_cols if c in df_pred.columns]].copy()

# 4. 데이터 최적화 다운캐스팅 (메모리 다이어트 및 성능 가산점 핵심 요소)
if 'predicted_churn' in df_pred_clean.columns:
    df_pred_clean['predicted_churn'] = df_pred_clean['predicted_churn'].astype(np.int8)  # 0, 1만 있으므로 int8로 충분
if 'risk_decile' in df_pred_clean.columns:
    df_pred_clean['risk_decile'] = df_pred_clean['risk_decile'].astype(np.int8)          # 1~10 수치이므로 int8로 충분
if 'churn_probability' in df_pred_clean.columns:
    df_pred_clean['churn_probability'] = df_pred_clean['churn_probability'].astype(np.float32) # 확률값은 float32로 소수점 다이어트
if 'risk_group' in df_pred_clean.columns:
    df_pred_clean['risk_group'] = df_pred_clean['risk_group'].astype('category')        # 그룹명은 카테고리형 변환

# 5. 고객_ID 기준으로 Left Join 병합 (기존 test 데이터 기준 픽스)
# 만약 기존 test 데이터에 이미 이전 버전의 예측 컬럼이 섞여있다면 꼬이지 않게 먼저 드롭해 줍니다.
dup_cols = ['predicted_churn', 'churn_probability', 'risk_decile', 'risk_group']
df_test = df_test.drop(columns=[c for c in dup_cols if c in df_test.columns], errors='ignore')

df_final = pd.merge(df_test, df_pred_clean, on='고객_ID', how='left')

# 6. 혹시 병합 과정에서 결측치(NaN)가 발생할 경우를 위한 최종 안전 초조치
if 'predicted_churn' in df_final.columns:
    df_final['predicted_churn'] = df_final['predicted_churn'].fillna(0).astype(np.int8)
if 'risk_decile' in df_final.columns:
    df_final['risk_decile'] = df_final['risk_decile'].fillna(10).astype(np.int8) # 결측치는 최하위 분위수 부여
if 'churn_probability' in df_final.columns:
    df_final['churn_probability'] = df_final['churn_probability'].fillna(0.0).astype(np.float32)
if 'risk_group' in df_final.columns:
    df_final['risk_group'] = df_final['risk_group'].fillna('Low Risk')

🚀 데이터셋 병합 및 최적화(다운캐스팅) 전처리를 시작합니다...


In [162]:
df_final.head()

,고객_ID,이탈여부,거주도시,가입채널,최초가입날짜,가입연도,가입월,가입일(일),가입요일,가입기간(일),...,완독 청취 곡수(100%),고유 재생 곡수,월간접속일수,마지막청취일,총청취시간(분),요금제_그룹,predicted_churn,churn_probability,risk_decile,risk_group
0,user_000001,0,1,7,2016-04-18,2016,4,18,월요일,2016.299315,...,2507,2177,31,20170331,11498,정기구독군,1,1.000000,1,High Risk
1,user_000002,0,18,7,2012-08-31,2012,8,31,금요일,2012.668265,...,836,796,16,20170326,3525,실속할인군,1,1.000000,1,High Risk
2,user_000003,0,1,7,2016-02-26,2016,2,26,금요일,2016.154566,...,1841,635,31,20170331,6737,실속할인군,1,1.000000,1,High Risk
3,user_000004,0,1,7,2013-08-17,2013,8,17,토요일,2013.629909,...,348,384,20,20170331,1811,정기구독군,1,1.000000,1,High Risk
4,user_000005,0,8,4,2015-11-18,2015,11,18,수요일,2015.882648,...,921,1027,30,20170331,4106,정기구독군,1,0.999999,1,High Risk


In [164]:
import numpy as np

# 1. 데이터셋 내 실제 정답 컬럼 유연하게 감지
actual_col = 'is_churn' if 'is_churn' in df_final.columns else ('이탈여부' if '이탈여부' in df_final.columns else None)

if actual_col and 'predicted_churn' in df_final.columns:
    # 2. 정답과 예측이 같으면 '적중', 다르면 '실패'로 컬럼 생성!
    df_final['예측적중_여부'] = np.where(df_final['predicted_churn'] == df_final[actual_col], '적중', '실패')
    
    # (선택) 시각화나 계산에 편리하게 1(성공)과 0(실패) 수치형으로도 만들고 싶다면:
    # df['예측적중_코드'] = (df['predicted_churn'] == df[actual_col]).astype(int)
    
    print("✅ 예측적중_여부 컬럼이 성공적으로 생성되었습니다!")
else:
    print("⚠️ 정답 컬럼 또는 predicted_churn 컬럼이 데이터셋에 존재하지 않습니다.")

✅ 예측적중_여부 컬럼이 성공적으로 생성되었습니다!


In [165]:
# 7. 원본 파일에 깨끗하게 덮어쓰기 저장
df_final.to_csv(test_path, index=False, encoding="utf-8-sig")

print(f"🎯 전처리 완료! 예측 정보가 결합된 마스터 마트가 덮어쓰기 되었습니다.")
print(f"📊 최종 데이터셋 구조: {df_final.shape[0]}행, {df_final.shape[1]}개 컬럼")

🎯 전처리 완료! 예측 정보가 결합된 마스터 마트가 덮어쓰기 되었습니다.
📊 최종 데이터셋 구조: 172194행, 49개 컬럼


In [166]:
df_final.head()

,고객_ID,이탈여부,거주도시,가입채널,최초가입날짜,가입연도,가입월,가입일(일),가입요일,가입기간(일),...,고유 재생 곡수,월간접속일수,마지막청취일,총청취시간(분),요금제_그룹,predicted_churn,churn_probability,risk_decile,risk_group,예측적중_여부
0,user_000001,0,1,7,2016-04-18,2016,4,18,월요일,2016.299315,...,2177,31,20170331,11498,정기구독군,1,1.000000,1,High Risk,실패
1,user_000002,0,18,7,2012-08-31,2012,8,31,금요일,2012.668265,...,796,16,20170326,3525,실속할인군,1,1.000000,1,High Risk,실패
2,user_000003,0,1,7,2016-02-26,2016,2,26,금요일,2016.154566,...,635,31,20170331,6737,실속할인군,1,1.000000,1,High Risk,실패
3,user_000004,0,1,7,2013-08-17,2013,8,17,토요일,2013.629909,...,384,20,20170331,1811,정기구독군,1,1.000000,1,High Risk,실패
4,user_000005,0,8,4,2015-11-18,2015,11,18,수요일,2015.882648,...,1027,30,20170331,4106,정기구독군,1,0.999999,1,High Risk,실패


In [181]:
shap = pd.read_csv("D:/skn31_2nd_pr/data/lgbm_feature_importance.csv")

In [182]:
shap.head()

,Feature,Importance
0,만료일(일),11449
1,가입기간(일),11107
2,스킵 청취 곡수(25% 미만),9458
3,결제일(일),8733
4,고유 재생 곡수,8590


In [183]:
import pandas as pd

def translate_features(df_feat):
    """
    SHAP/Feature Importance처럼 영문 변수명이 데이터 '행(Value)' 값으로 
    박혀있을 때 이를 발표 규격 한글로 치환해 주는 최종 철벽 함수
    """
    feature_map = {
        "trans_day": "결제일(일)", "expire_day": "만료일(일)",
        "trans_weekday": "결제요일", "expire_weekday": "만료요일",
        "trans_month": "결제월", "expire_month": "만료월",
        "trans_year": "결제연도", "expire_year": "만료연도",
        "reg_time_num": "가입기간(일)", "reg_day": "가입일(일)",
        "reg_month": "가입월", "reg_year": "가입연도", "reg_weekday": "가입요일",
        "city": "거주도시", "registered_via": "가입채널",
        "num_25": "스킵 청취 곡수(25% 미만)", "num_50": "단기 청취 곡수(50%)",
        "num_75": "중기 청취 곡수(75%)", "num_985": "장기 청취 곡수(98.5%)",
        "num_100": "완독 청취 곡수(100%)", "num_unq": "고유 재생 곡수",
        "total_secs": "총청취시간(초)", "log_count": "월간접속일수",
        "last_listen_date": "마지막청취일", "payment_method_id": "결제수단ID",
        "trans_count": "누적결제수", "is_auto_renew": "자동갱신_동의여부",
        "plan_list_price": "구독 정가", "actual_amount_paid": "실제_결제금액",
        "is_cancel": "구독_직접_취소여부", "tx_payment_plan_days": "플랜이용기간",
        "gender_male": "성별: 남성", "gender_female": "성별: 여성", "gender_unknown": "성별: 미입력",
        "pm_id_38": "결제수단ID_31번", "pm_id_41": "결제수단ID_34번",
        "pp_days_30": "30일_정기구독", "is_churn":"이탈여부", "registration_init_time":"최초가입날짜",
        "membership_expire_date":"멤버십종료일", "transaction_date":"가장최근결제날짜", "payment_plan_days":"구독플랜일수"
    }
    
    df_clean = df_feat.copy()
    
    # 🔍 방어선 1: 대소문자 무관하게 변수명 컬럼 위치 찾아내기
    # 데이터셋의 컬럼 중 'Feature', 'feature' 성향을 띠는 컬럼을 타겟팅합니다.
    target_col = None
    for col in df_clean.columns:
        if col.strip().lower() == 'feature':
            target_col = col
            break
            
    if target_col is not None:
        # 데이터 공백 싹 지우기(strip)
        df_clean[target_col] = df_clean[target_col].astype(str).str.strip()
        # 🎯 .map()을 사용하여 행 내부 데이터를 딕셔너리 기반으로 치환 (매핑 없는 건 원본 유지)
        df_clean[target_col] = df_clean[target_col].map(feature_map).fillna(df_clean[target_col])
    else:
        # 혹시 컬럼 대가리 자체도 영문일 수 있으므로 기존 컬럼명 rename도 백업으로 작동
        df_clean.rename(columns=feature_map, inplace=True)
        
    return df_clean



In [179]:
shap_ko = translate_features(shap)

shap_ko['Feature']

0               만료일(일)
1              가입기간(일)
2     스킵 청취 곡수(25% 미만)
3               결제일(일)
4             고유 재생 곡수
5                누적결제수
6               가입일(일)
7        단기 청취 곡수(50%)
8       완독 청취 곡수(100%)
9             총청취시간(초)
10     장기 청취 곡수(98.5%)
11                만료요일
12       중기 청취 곡수(75%)
13              월간접속일수
14                 가입월
15                거주도시
16                결제요일
17                가입요일
18              결제수단ID
19                가입채널
20               구독 정가
21                가입연도
22           자동갱신_동의여부
23             실제_결제금액
24          구독_직접_취소여부
25              성별: 남성
26              성별: 여성
27             성별: 미입력
28          결제수단ID_31번
29              구독플랜일수
30          결제수단ID_34번
31            30일_정기구독
Name: Feature, dtype: str

In [ ]:
# 7. 원본 파일에 깨끗하게 덮어쓰기 저장
file_path = "D:/skn31_2nd_pr/data/lgbm_feature_importance.csv"
shap_ko.to_csv(file_path, index=False, encoding="utf-8-sig")

print(f"🎯 전처리 완료! 예측 정보가 결합된 마스터 마트가 덮어쓰기 되었습니다.")
print(f"📊 최종 데이터셋 구조: {shap_ko.shape[0]}행, {shap_ko.shape[1]}개 컬럼")

🎯 전처리 완료! 예측 정보가 결합된 마스터 마트가 덮어쓰기 되었습니다.
📊 최종 데이터셋 구조: 32행, 2개 컬럼


<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Playdata\AppData\Local\Temp\ipykernel_32064\1408763483.py:2: SyntaxWarning: invalid escape sequence '\s'
  file_path = "D:\skn31_2nd_pr\data\lgbm_feature_importance.csv"


In [175]:
shap_ko.head()

,Feature,mean_abs_shap
0,만료일(일),0.810931
1,자동갱신_동의여부,0.751060
2,만료요일,0.666224
3,결제일(일),0.637801
4,구독_직접_취소여부,0.396002
